In [ ]:
# GP (2026-08-21) stuff done in SNT25-618
# - Change issue as not really duplicated ... https://bluesquare.atlassian.net/browse/SNT25-618 is SMALLER than the older one (JUST NOT BREAK AND FIGS SHOW UP OK)
# - Code replaced by utils funs: a few more to be adde, but that requiers more work on SNT_utils.R (separate Jira)
# - Cleaned up text and translated to French
# - Updated a breaking hardcoded parameter `USE_ADJUSTED_POPULATION` -> `USE_TRANSFORMED_POPULATION`

# Remains ToDo:
# - [in CODE nb!] Adj3 should be NA if Adj2 is NA ... ! 
#   -> see: https://bluesquare.atlassian.net/browse/SNT25-646
# - 🚨 Need to handle cases where `FLAG_*` is (all) `NA`s ... !!
#       (this should be fixed by above point)
# - Plots of RR should be MOVED to the RR pipeline report (not incidence)!!!
#   (these were included here because incidence consumes RR and therefore RR affects results,
#   but the RR plots are not really part of incidence reporting)
# - Scatter plots (coherence) should be made with 📦{scattermore} (rasterized) for lighter figures

In [ ]:
# Code tested in:
# - "SNT Testing" (COD)
# - "NER SNT Process" (NER)
# - "CMR SNT Process" (CMR)
# - "BDI SNT Process" (BDI)

# Estimations de l’incidence brute et ajustée

## 1. Setup

In [ ]:
SNT_ROOT_PATH  <- "~/workspace"
CODE_PATH      <- file.path(SNT_ROOT_PATH, "code")
CONFIG_PATH    <- file.path(SNT_ROOT_PATH, "configuration")
DATA_PATH <- file.path(SNT_ROOT_PATH, 'data', 'dhis2', 'incidence') # output of the pipeline (only final results)

INTERMEDIATE_DATA_PATH <- file.path(DATA_PATH, "intermediate_results") # for reporting nb or else, NOT for OH Dataset!

REPORTING_NB_PATH <- file.path(SNT_ROOT_PATH, "pipelines/snt_dhis2_incidence/reporting")
FIGURES_PATH <- file.path(REPORTING_NB_PATH, "outputs", "figures")

In [ ]:
source(file.path(CODE_PATH, "snt_utils.r"))
source(file.path(CODE_PATH, "snt_palettes.r"))

In [ ]:
required_packages <- c(
    "dplyr",
    "tidyr",
    "ggplot2",
    "stringr",
    "glue",
    "arrow",
    "sf",
    "reticulate" 
    )

install_and_load(required_packages)

In [ ]:
# Create output directories if they don't exist (needs utils)
safe_create_dir(FIGURES_PATH) |> suppressMessages()

In [ ]:
# ⚠️🧹 GP: Move this function to ./code/snt_utils.r !
# (do as separate branch/task, see https://bluesquare.atlassian.net/browse/SNT25-591 )
init_openhexa_env <- function(
  # In nb, run as `openhexa <- init_openhexa_env()`
  python_path = "/opt/conda/bin/python",
  proj_lib = "/opt/conda/share/proj",
  gdal_data = "/opt/conda/share/gdal"
) {
  Sys.setenv(PROJ_LIB = proj_lib)
  Sys.setenv(GDAL_DATA = gdal_data)
  Sys.setenv(RETICULATE_PYTHON = python_path)
  reticulate::py_config()$python

  return(reticulate::import("openhexa.sdk"))
}

openhexa <- init_openhexa_env()

#### Load `SNT_config`

In [ ]:
config_json <- load_snt_config(config_path = CONFIG_PATH)

In [ ]:
# Configuration variables
DATASET_NAME <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_INCIDENCE
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE

# Cols to select from pyramid
ADMIN_1_NAME <- toupper(config_json$SNT_CONFIG$DHIS2_ADMINISTRATION_1)
ADMIN_2_NAME <- toupper(config_json$SNT_CONFIG$DHIS2_ADMINISTRATION_2)
ADMIN_1_ID <- str_replace(ADMIN_1_NAME, "_NAME", "_ID")
ADMIN_2_ID <- str_replace(ADMIN_2_NAME, "_NAME", "_ID")

#### Load `SNT_metadata`
This is needed for the correct use of palettes and categories (breaks, or scale)

In [ ]:
# ⚠️ GP (2026-08-20): 
# - Use utils fun to import 
# - Consider removing dependency on "SNT_metadata.json" ... !

# Load SNT metadata
metadata_json <- tryCatch({ jsonlite::fromJSON(file.path(CONFIG_PATH, "SNT_metadata.json")) },
    error = function(e) {
        msg <- paste0("[ERROR] Error while loading metadata", conditionMessage(e))  
        cat(msg)   
        stop(msg) 
    })

log_msg(paste0("SNT metadata loaded from : ", file.path(CONFIG_PATH, "SNT_metadata.json")))

In [ ]:
# Handle situation in which metadata is not correctly loaded (for example if the specific node is missing in the json file)
if (is.null(metadata_json$INCIDENCE_CRUDE$SCALE)) {
    log_msg("Info: Incidence (crude) scale break values cannot be loaded from SNT_metadata.json because the node $INCIDENCE_CRUDE$SCALE is missing.")
} else {
    break_vals <- jsonlite::fromJSON(metadata_json$INCIDENCE_CRUDE$SCALE)
    log_msg(paste0("Incidence (crude) scale break values loaded from SNT_metadata.json : ", paste(break_vals, collapse = ", ")))
}

## 2. Load data

In [ ]:
DATASET_DHIS2 <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED
DATASET_INCIDENCE <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_INCIDENCE

#### 2.0. Parameters: `parameters_json`

In [ ]:
parameters <- load_country_file_from_dataset(
    dataset_id = DATASET_INCIDENCE, 
    country_code = COUNTRY_CODE, 
    suffix = "_parameters.json" 
    )

# Dynamically assign each parameter as a variable named after its key
list2env(parameters, envir = environment())

# Display values in report (just to check)
glimpse(parameters)

#### 2.1. Shapes

In [ ]:
shapes_data <- load_country_file_from_dataset(
    dataset_id = DATASET_DHIS2, 
    country_code = COUNTRY_CODE, 
    suffix = "_shapes.geojson" 
    )

# printdim(shapes_data) needs function from utils (currently not in snt_utils.r)
dim(shapes_data)

In [ ]:
# Simplify shapes to make it easier to render (fewer points per polygon)
# Set dTolerance bigger to keep fewer points
shapes_data_stsimplified <- st_simplify(shapes_data, dTolerance = 1000) 
# GP:make this more refined to somehow reduce points without losing too much shape 
# (maybe use `rmapshaper::ms_simplify()` instead of `st_simplify()`) ... ?

# ⚠️ GP: Silence this as user does not need to read this in the report ...
print("Simplifying shapes for faster rendering ...")
print(paste0("Original shapes data points: ", sum(mapply(function(g) length(unlist(g)), sf::st_geometry(shapes_data))))) 
print(paste0("Simplified shapes data points: ", sum(mapply(function(g) length(unlist(g)), sf::st_geometry(shapes_data_stsimplified)))))

#### 2.2. Pyramid

In [ ]:
# This is needed to add back the `*_NAME` cols to the main data <br>
# (Because normally we only output tables with the `*_ID` cols)

In [ ]:
pyramid_data <- load_country_file_from_dataset(
    dataset_id = DATASET_DHIS2, 
    country_code = COUNTRY_CODE, 
    suffix = "_pyramid.parquet" 
    )

# head(pyramid_data, 3)
dim(pyramid_data)

In [ ]:
# Keep only relevant cols and rename them to match incidence data
pyramid <- pyramid_data %>%
  select(
    ADM1_ID = all_of(ADMIN_1_ID),
    ADM1_NAME = all_of(ADMIN_1_NAME), 
    ADM2_ID = all_of(ADMIN_2_ID),
    ADM2_NAME = all_of(ADMIN_2_NAME)
  ) %>%
  distinct()

# head(pyramid, 3)

#### 2.3. Cas par mois
Nécessaire pour les vérifications de **cohérence**:
* **TPR** au niveau mensuel au fil du temps
    * Expliquer les changements (ou l'absence de changement) entre Brut et Adj1
    * Utile pour surveiller la résistance (ou le comportement de dépistage... ?)
* **Reporting Rate**
    * Expliquer les changements (ou l'absence de changement) entre Adj1 et Adj2
* Cohérence des **indicateurs** :
    * SUSP > TEST
    * TEST > CONF
    * ... (vérifier et en ajouter d'autres...)

In [ ]:
# ⚠️ Note: **Import** from 📁`/data/` folder (not OH Dataset) <br>

In [ ]:
file_path <- file.path(INTERMEDIATE_DATA_PATH, paste0(COUNTRY_CODE, "_monthly_cases.parquet"))
monthly_cases <- arrow::read_parquet(file_path)
log_msg(paste0("Monthly cases data loaded from : ", file_path))

dim(monthly_cases)
head(monthly_cases, 3)

In [ ]:
# Add _NAME cols by joining with pyramid_data
monthly_cases <- left_join(monthly_cases, pyramid, by = join_by(ADM1_ID, ADM2_ID))

In [ ]:
# head(monthly_cases, 3)

#### 2.4. Incidence Annuelle

In [ ]:
# **Note**: `REPORTING_RATE_METHOD` this is NOT a parameter!<br>
# The method is derived based on what is available in the dataset `config_json$SNT_DATASET_IDENTIFIERS$DHIS2_REPORTING_RATE`

In [ ]:
# Define dataset and file names (based on parameter)
rr_dataset_name <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_REPORTING_RATE
file_name_de <- paste0(COUNTRY_CODE, "_reporting_rate_dataelement.parquet")
file_name_ds <- paste0(COUNTRY_CODE, "_reporting_rate_dataset.parquet")

# Determine REPORTING_RATE_METHOD based on available file names in the dataset (without loading files)
dataset_last_version <- openhexa$workspace$get_dataset(rr_dataset_name)$latest_version
files_iter <- dataset_last_version$files

files <- list()
repeat {
  file <- tryCatch(
    py_to_r(iter_next(files_iter)),
    error = function(e) NULL
  )
  if (is.null(file)) break
  files <- append(files, list(file))
}

filenames <- sapply(files, function(f) f$filename)

if (file_name_de %in% filenames) {
  REPORTING_RATE_METHOD <- "dataelement"
} else if (file_name_ds %in% filenames) {
  REPORTING_RATE_METHOD <- "dataset"
} else {
  stop(glue("[ERROR] Neither reporting rate file found for: {COUNTRY_CODE}"))
}

log_msg(paste0("Determined REPORTING_RATE_METHOD: ", REPORTING_RATE_METHOD))

In [ ]:
yearly_incidence <- load_country_file_from_dataset(
    dataset_id = DATASET_NAME, 
    country_code = COUNTRY_CODE, 
    suffix = "_incidence.parquet" 
    )

dim(yearly_incidence)
head(yearly_incidence, 3)

In [ ]:
## Plot settings
### 🎨 Dynamic categories and color assignement

In [ ]:
# Define suffix for exporting final outputs (preserve selected disaggregation in filename)
DISAGGREGATION_SELECTION_SUFFIX <- ifelse(is.null(DISAGGREGATION_SELECTION), "TOTAL", DISAGGREGATION_SELECTION)

In [ ]:
##### 1. Define breaks and labels

In [ ]:
# Safety code to avoid breaking if nothings is fund in json_metadata
if (!exists("break_vals") || is.null(break_vals) || length(break_vals) == 0) {
    # log_msg("[WARNING] No break values found in SNT_metadata.json for INCIDENCE_CRUDE$SCALE. Using default values.", "warning")
    break_vals <- c(100, 250, 450, 1000)
        log_msg(glue::glue("No break values found in SNT_metadata.json for INCIDENCE_CRUDE$SCALE. Using default values: {paste(break_vals, collapse = ', ')}"), 
        "info")
}

In [ ]:
# Create the full set of cut points (0 to Infinity)
full_breaks <- c(0, break_vals, Inf)

# Create dynamic labels
labels <- c(
  paste0("< ", break_vals[1]),                                      # First label
  paste0(break_vals[-length(break_vals)], "-", break_vals[-1]),     # Middle labels
  paste0("> ", break_vals[length(break_vals)])                       # Last label
)

# Check
# labels

##### 3. Pick appropriate palette

In [ ]:
# Count nr of breaks
nr_of_colors <- length(labels)

# nr_of_colors
palette_to_use <- get_range_from_count(nr_of_colors)

# # Need to make palettes as named vectors so that scale_color_manual() and scale_fill_manual() can use them properly
# # Note: need to reverse order of labels to match the palette order "meaning" (red "" should correcpond to lowest value)
# names(palette_to_use) <- rev(labels)

print(palette_to_use)


In [ ]:
# ⚠️ TEMP fix to handle older parameter `USE_ADJUSTED_POPULATION`
#  now replaced by `USE_TRANSFORMED_POPULATION`
if (exists("USE_ADJUSTED_POPULATION")) {
    USE_TRANSFORMED_POPULATION <- USE_ADJUSTED_POPULATION
}

In [ ]:
# Repeated text embedded in plots
plot_caption_n1 <- glue::glue("Méthode de calcul de N1: {N1_METHOD}.")
plot_caption_adjpop <- glue::glue("Utilisation de la population ajustée: {USE_TRANSFORMED_POPULATION}.")
plot_caption_routine <- glue::glue("Données de routine: {ROUTINE_DATA_CHOICE}.")
plot_caption_reporting_rate <- glue::glue("Taux de déclaration calculé selon la méthode : {REPORTING_RATE_METHOD}.")
plot_caption_csbfile <- glue::glue("Données CSB fournies par l'utilisateur (fichier) : {CARESEEKING_FILE_PATH}.")

plot_caption <- paste(
    plot_caption_n1, 
    plot_caption_adjpop, 
    plot_caption_routine, 
    plot_caption_reporting_rate, 
    plot_caption_csbfile, 
    sep = "\n"
)
# plot_caption

## Contrôles de cohérence

In [ ]:
# See Jira: https://bluesquare.atlassian.net/browse/SNT25-272

#### 1. TPR
Taux de Positivité des Tests

In [ ]:
# Calculate yearly TPR to be added on top of the monthly TPR plots
monthly_cases_yearly <- monthly_cases %>%
    group_by(ADM1_NAME, ADM2_ID, ADM2_NAME, YEAR) %>%  
    mutate(
        CONF_yearly = sum(CONF, na.rm = TRUE),
        TEST_yearly = sum(TEST, na.rm = TRUE)
    ) %>%
    ungroup() %>%
    mutate(
      TPR_yearly = ifelse(!is.na(CONF_yearly) & !is.na(TEST_yearly) & (TEST_yearly != 0), CONF_yearly / TEST_yearly, 1)
    ) 

# head(monthly_cases_yearly, 3)

##### 1.1. TPR (monthly) over time

In [ ]:
# ⚠️ TO BE MOVED TO UTILS (as a function) ...
# Code to calculate width and height of plots based on the number of unique years and ADM2 regions

nr_unique_year <- length(unique(monthly_cases_yearly$YEAR))
calc_width <- max(10, nr_unique_year * 5)

nr_unique_adm2 <- length(unique(monthly_cases_yearly$ADM2_ID))
# (max ggsave size: 50 inches = 127 cm)
calc_height <- min(126, ceiling(nr_unique_adm2 / 3))

In [ ]:
plot <- ggplot(monthly_cases_yearly) +
# Monthly TPR lines
  geom_hline(
    yintercept = 0,
    color = "grey21",
    linewidth = 0.5
  ) +
  geom_hline(
    yintercept = c(0.25, 0.5, 0.75, 1.0),
    color = "grey69",
    linewidth = 0.25
  ) +
  geom_line(
    aes(x = MONTH, y = TPR, group = ADM2_NAME),
    color = "grey21",
    alpha = 0.75) +
  facet_grid(
    cols = vars(YEAR), rows = vars(ADM1_NAME),
    switch = "y") +
  scale_x_continuous(breaks = seq(1,12,1)) +
  scale_y_continuous(labels = scales::percent_format(accuracy = 1L), limits = c(0, 1)) +
  labs(
    title = "Taux de Positivité des Tests (TPR) pour ADM2 et mois"  ) +
  theme_minimal() +
  theme(
    panel.grid.minor = element_blank(),
    panel.grid.major.y = element_blank(),
    strip.placement = "outside",
    strip.background = element_rect(fill = "grey21"),
    strip.text = element_text(color = "white"),
    axis.text.x = element_text(angle = 90, vjust = 0.5),
    axis.title.y = element_blank()
  )

# Export plot with ggsave to figures_dir
plot_dir = file.path(FIGURES_PATH, glue::glue("TPR_monthly_{DISAGGREGATION_SELECTION_SUFFIX}.png"))
ggsave(
  filename = plot_dir,
  plot = plot,
  width = calc_width, 
  height = calc_height, 
  units = "cm", 
  dpi = 200
) 

IRdisplay::display_png(file = plot_dir)

#### 2. RR

Taux de Rapportage.

Pour plus de détails, consultez les cahiers **de rapport** du pipeline "**A.4 DHIS2 Reporting Rate (Dataset)**" ou "**A.4 DHIS2 Reporting Rate (Data element)**", selon la méthode de calcul du taux de déclaration utilisée.<br>
Options possibles :
* **Dataset**: pipelines/snt_dhis2_reporting_rate_dataset/reporting/outputs/**snt_dhis2_reporting_rate_dataset_report**\_OUTPUT\_\*.ipynb
* **Data Element**: pipelines/snt_dhis2_reporting_rate_dataelement/reporting/outputs/**snt_dhis2_reporting_rate_dataelement_report**\_OUTPUT\_\*.ipynb

In [ ]:
# ⚠️ THIS SHOULD BE MOVED TO RR PIPELINE report ... !

In [ ]:
# ⚠️ **TO DO**: align code here with report notebook pf reporting rate (use "🎨 NEW dynamic colors & breaks" approach)

In [ ]:
# Tile plot faceted by YEAR
plot <- ggplot(data = monthly_cases) +
  geom_tile(aes(x = MONTH,
                y = forcats::fct_rev(ADM2_NAME),
                # fill = REPORTING_RATE_CATEGORY
                fill = REPORTING_RATE
                ), 
                color = "white",
                show.legend = TRUE,
                # Fill NA values with white
                na.rm = FALSE
                ) +
#   scale_fill_manual(
#       values = palette_to_use, # 🎨 NEW dynamic colors & breaks!
#       na.value = "white",
#       name = "Reporting Rate: "
#     ) +
  scale_fill_viridis_c(
      option = "viridis",
      na.value = "white",
      name = "Reporting Rate:",
      direction = -1
      # labels = scales::percent_format(accuracy = 1L)
    ) +
  scale_x_continuous(breaks = seq(1, 12, 1)) +
  facet_grid(rows = vars(ADM1_NAME), cols = vars(YEAR), 
    scales = "free_y", space = "free_y",
    switch = "y") +
  theme_minimal() +
  theme(
    plot.subtitle = element_text(margin=margin(0,0,20,0)),
    legend.position = "bottom",
    legend.key.height = unit(0.25, "cm"),
    axis.text.x = element_text(size = 7, angle = 90, vjust = 0.5, hjust = 1),
    axis.text.y = element_text(size = 7),
    axis.title.y = element_blank(),
    panel.grid.minor = element_blank(),
    panel.grid.major = element_blank(),
    strip.placement = "outside",    
    strip.text = element_text(color = "white", face = "bold", size = 10),
    strip.background = element_rect(fill = "grey21")
  ) +
  guides(fill = guide_legend(nrow = 1))

# Export plot with ggsave to figures_dir
plot_dir = file.path(FIGURES_PATH, glue::glue("ReportingRate_heatmap_monthly_{DISAGGREGATION_SELECTION_SUFFIX}.png"))
ggsave(
  filename = plot_dir,
  plot = plot,
  width = calc_width, 
  height = calc_height, 
  units = "cm", 
  dpi = 200
) 

IRdisplay::display_png(file = plot_dir)

In [ ]:
# # Check on data completeness for REPORTING RATE data: 
# # check how many values (and what proportion) of REPORTING_RATE are NA
# na_count <- sum(is.na(monthly_cases$REPORTING_RATE))
# na_prop <- na_count / nrow(monthly_cases)
# if (na_count > 0) {
#     log_msg(glue("⚠️ Warning: Reporting Rate data contains {na_count} missing values (NA) in 'REPORTING_RATE' column ({scales::percent(na_prop, accuracy = 0.1)})."), "warning")
# } else {
#     log_msg("✅ Reporting Rate data contains no missing values (NA) in 'REPORTING_RATE' column.")
# }

In [ ]:
# Same check, broken down per YEAR
na_prop_by_year <- monthly_cases %>%
    group_by(YEAR) %>%
    summarise(
        na_count = sum(is.na(REPORTING_RATE)),
        n = n(),
        na_prop = na_count / n,
        .groups = "drop"
    )

for (i in seq_len(nrow(na_prop_by_year))) {
    row <- na_prop_by_year[i, ]
    if (row$na_count > 0) {
        print(glue("⚠️ Warning: Year {row$YEAR} - Reporting Rate data contains {row$na_count} missing values (NA) in 'REPORTING_RATE' column ({scales::percent(row$na_prop, accuracy = 0.1)})."))
    } else {
        print(glue("✅ Year {row$YEAR} - Reporting Rate data contains no missing values (NA) in 'REPORTING_RATE' column."))
    }
}

### 3. Contrôles de cohérence sur l'incidence : Graphique en nuage de points

Logique : chaque niveau d'ajustement doit produire des valeurs supérieures (ou égales) à celles du niveau précédent.<br>

Plus précisément:
* Brut <= Adj1
* Ajusté1 <= Ajusté2
* Ajusté2 <= Ajusté3

Étant donné que les valeurs brutes, Ajusté1, Ajusté2 et Ajusté3 sont calculées en agrégeant `CONF`, `N1`, `N2` et `N3` au niveau ADM2 x ANNÉE, nous pouvons d’abord vérifier que la relation entre ces valeurs est cohérente. À propos, vérifier si
* `CONF` <= `N1`
* `N1` ≤ `N2`
* `N2` ≤ `N3`

#### 3.1. Mesures d'incidence
Mesures utilisées pour calculer l'incidence: `CONF`, `N1`, `N2`, (et `N3`)

In [ ]:
# CONF vs N1 

# Create warning message if there are CONF values greater than N1
conf_greater_n1_count <- sum(monthly_cases$CONF > monthly_cases$N1, na.rm = TRUE)
if (conf_greater_n1_count > 0) {
    warning_text <- glue("✘ Warning: There are {conf_greater_n1_count} instances where CONF is greater than N1.", "warning")
} else {
    warning_text <- "✔ All CONF values are less than or equal to N1."
}

plot <- ggplot(data = monthly_cases) +
  geom_abline(intercept = 0, slope = 1, linetype = "dashed", color = "red") +
  geom_point(
    aes(
      x = N1,
      y = CONF),
    alpha = 0.5) +
  labs(
    title = "CONF vs N1",
    subtitle = "N1 is expected to be greater or equal to CONF",
    caption = warning_text
    ) +
  theme_minimal(base_size = 7) +
  theme(
    aspect.ratio = 1,
    plot.caption.position = "plot",
    plot.caption = element_text(hjust = 0)
  )

plot_dir <- file.path(FIGURES_PATH, glue::glue("CONF_vs_N1_{DISAGGREGATION_SELECTION_SUFFIX}.png"))
ggsave(
  filename = plot_dir, 
  plot = plot, 
  # width = 21, 
  # height = 21, 
  width = 10, 
  height = 10, 
  units = "cm", 
  dpi = 200
  )

IRdisplay::display_png(file = plot_dir)

In [ ]:
# N1 > N2

# Create warning message if there are N1 values greater than N2
n1_greater_n2_count <- sum(monthly_cases$N1 > monthly_cases$N2, na.rm = TRUE)
if (n1_greater_n2_count > 0) {
    warning_text <- glue("✘ Warning: There are {n1_greater_n2_count} instances where N1 is greater than N2.", "warning")
} else {
    warning_text <- "✔ All N1 values are less than or equal to N2."
}

plot <- ggplot(data = monthly_cases) +
  geom_abline(intercept = 0, slope = 1, linetype = "dashed", color = "red") +
  geom_point(
    aes(
      x = N2,
      y = N1),
    alpha = 0.5) +
  labs(title = "N1 vs N2",
       subtitle = "N2 is expected to be greater or equal to N1.",
       caption = warning_text
       ) +
  theme_minimal(base_size = 7) +
  theme(
    aspect.ratio = 1,
    plot.caption.position = "plot",
    plot.caption = element_text(hjust = 0)
  )

plot_dir <- file.path(FIGURES_PATH, glue::glue("N1_vs_N2_{DISAGGREGATION_SELECTION_SUFFIX}.png"))
ggsave(
  filename = plot_dir, 
  plot = plot, 
  width = 10, 
  height = 10, 
  units = "cm", 
  dpi = 200
  )

IRdisplay::display_png(file = plot_dir)

#### 3.2. Incidence values
Actual (calculated) incidence: Crude, Adj1, Adj2, Adj3

In [ ]:
# Add col to mark cases where INCIDENCE_ADJ_TESTING < INCIDENCE_CRUDE so that it is displayed in red in the plot
yearly_incidence_plot <- yearly_incidence %>%
  mutate(
    FLAG_CRUDE_VS_ADJTEST = ifelse(INCIDENCE_ADJ_TESTING < INCIDENCE_CRUDE, TRUE, FALSE),
    FLAG_ADJTEST_VS_ADJREP = ifelse(INCIDENCE_ADJ_REPORTING < INCIDENCE_ADJ_TESTING, TRUE, FALSE)
  )

if ("INCIDENCE_ADJ_CARESEEKING" %in% colnames(yearly_incidence) && any(!is.na(yearly_incidence$INCIDENCE_ADJ_CARESEEKING))) {
    # Create col to flag cases where INCIDENCE_ADJ_TESTING > INCIDENCE_ADJ_CARESEEKING
    yearly_incidence_plot <- yearly_incidence_plot %>%
      mutate(
        FLAG_ADJTEST_VS_ADJCARE = ifelse(INCIDENCE_ADJ_TESTING > INCIDENCE_ADJ_CARESEEKING, TRUE, FALSE)
      )
}

# head(yearly_incidence_plot)

##### Crude vs Adj for Testing (Adj1)

In [ ]:
# Create warning message if there are INCIDENCE_CRUDE values greater than INCIDENCE_ADJ_TESTING
incidence_crude_greater_adj1_count <- sum(yearly_incidence_plot$FLAG_CRUDE_VS_ADJTEST, na.rm = TRUE)  
if (incidence_crude_greater_adj1_count > 0) {
    warning_text <- glue("✘ Attention: il y a {incidence_crude_greater_adj1_count} instances où INCIDENCE_CRUDE est supérieure à INCIDENCE_ADJ_TESTING.", "warning")
} else {
    warning_text <- "✔ Toutes les valeurs INCIDENCE_CRUDE sont inférieures ou égales à INCIDENCE_ADJ_TESTING."
}

# Plot with points colored based on FLAG_CRUDE_VS_ADJTEST and faceted by YEAR
plot <- ggplot(data = yearly_incidence_plot) +
  geom_abline(intercept = 0, slope = 1, linetype = "dashed", color = "black") +
  geom_point(
    aes(
      x = INCIDENCE_CRUDE,
      y = INCIDENCE_ADJ_TESTING,
      color = FLAG_CRUDE_VS_ADJTEST),
    alpha = 0.7,
    size = 2) +
  scale_color_manual(
    values = c("FALSE" = "black", "TRUE" = "red")
  ) +
  scale_x_continuous(limits = c(0, NA), breaks = c(0, break_vals)) +
  scale_y_continuous(limits = c(0, NA), breaks = c(0, break_vals)) +
  facet_wrap(vars(YEAR), nrow = 1) +
  labs(
    title = "INCIDENCE_CRUDE vs INCIDENCE_ADJ_TESTING",
    subtitle = warning_text,
    caption = plot_caption
    ) +
  theme_minimal(base_size = 7) +
  theme(
    aspect.ratio = 1,
    legend.position = "none",
    strip.text = element_text(face = "bold", size = 10),
    axis.text.x = element_text(angle = 90),
    panel.grid.minor = element_blank(),
    plot.caption = element_text(size = 7, hjust = 0)
  )

plot_dir <- file.path(FIGURES_PATH, glue::glue("Incidence_year_crude_vs_adj_testing_{DISAGGREGATION_SELECTION_SUFFIX}.png"))
ggsave(
  filename = plot_dir, 
  plot = plot, 
  width = calc_width, 
  height = 21 / 2, 
  units = "cm", 
  dpi = 300
  )

IRdisplay::display_png(file = plot_dir)

##### Adj for Testing (Adj1) vs Adj for Reporting (Adj2)

In [ ]:
# 🚨 Need to handle cases where `FLAG_*` is (all) `NA`s ... !!

In [ ]:
# Create warning message if there are INCIDENCE_ADJ_TESTING values greater than INCIDENCE_ADJ_REPORTING
incidence_adj1_greater_adj2_count <- sum(yearly_incidence_plot$FLAG_ADJTEST_VS_ADJREP, na.rm = TRUE) 
if (incidence_adj1_greater_adj2_count > 0) {
    warning_text <- glue("✘ Attention: il y a {incidence_adj1_greater_adj2_count} instances où INCIDENCE_ADJ_TESTING est supérieure à INCIDENCE_ADJ_REPORTING.", "warning")
} else {
    warning_text <- "✔ Toutes les valeurs INCIDENCE_ADJ_TESTING sont inférieures ou égales à INCIDENCE_ADJ_REPORTING."
}

# Plot with points colored based on FLAG_ADJTEST_VS_ADJREP and faceted by YEAR
plot <- ggplot(data = yearly_incidence_plot) +
    geom_abline(intercept = 0, slope = 1, linetype = "dashed", color = "black") +
    geom_point(
        aes(
        x = INCIDENCE_ADJ_TESTING,
        y = INCIDENCE_ADJ_REPORTING,
        color = FLAG_ADJTEST_VS_ADJREP),
        alpha = 0.7,
        size = 2) +
    scale_color_manual(
        values = c("FALSE" = "black", "TRUE" = "red")
    ) +
    scale_x_continuous(limits = c(0, NA), breaks = c(0, break_vals)) +
    scale_y_continuous(limits = c(0, NA), breaks = c(0, break_vals)) +
    facet_wrap(vars(YEAR), nrow = 1) +
    labs(
        title = "INCIDENCE_ADJ_TESTING vs INCIDENCE_ADJ_REPORTING",
        subtitle = warning_text,
        caption = plot_caption
        ) +
    theme_minimal(base_size = 7) +
    theme(
        aspect.ratio = 1,
        legend.position = "none",
        strip.text = element_text(face = "bold", size = 10),
        axis.text.x = element_text(angle = 90),
        panel.grid.minor = element_blank(),
        plot.caption = element_text(size = 7, hjust = 0)
    )

plot_dir <- file.path(FIGURES_PATH, glue::glue("Incidence_year_adj_testing_vs_adj_reporting_{DISAGGREGATION_SELECTION_SUFFIX}.png"))
ggsave(
  filename = plot_dir, 
  plot = plot, 
  width = calc_width, 
  height = 21 / 2, 
  units = "cm", 
  dpi = 300
  )

IRdisplay::display_png(file = plot_dir)

##### Adj for Reporting (Adj2) vs Adj for Care Seeking Behaviour (Adj3)

In [ ]:
if ("INCIDENCE_ADJ_CARESEEKING" %in% colnames(yearly_incidence) && any(!is.na(yearly_incidence$INCIDENCE_ADJ_CARESEEKING))) {

    # Create warning message if there are INCIDENCE_ADJ_TESTING values greater than INCIDENCE_ADJ_CARESEEKING
    incidence_adj2_greater_adj3_count <- sum(yearly_incidence$INCIDENCE_ADJ_TESTING > yearly_incidence$INCIDENCE_ADJ_CARESEEKING, na.rm = TRUE)  
    if (incidence_adj2_greater_adj3_count > 0) {
       warning_text <- glue("✘ Attention: il y a {incidence_adj2_greater_adj3_count} instances où INCIDENCE_ADJ_TESTING est supérieure à INCIDENCE_ADJ_CARESEEKING.")
    } else {
       warning_text <- "✔ Toutes les valeurs INCIDENCE_ADJ_TESTING sont inférieures ou égales à INCIDENCE_ADJ_CARESEEKING."
    }

    # Plot with points colored based on FLAG_ADJTEST_VS_ADJREP and faceted by YEAR
    p <- ggplot(data = yearly_incidence_plot) +
        geom_abline(intercept = 0, slope = 1, linetype = "dashed", color = "black") +
        geom_point(
            aes(
            x = INCIDENCE_ADJ_TESTING,
            y = INCIDENCE_ADJ_CARESEEKING,
            color = FLAG_ADJTEST_VS_ADJCARE),
            alpha = 0.7,
            size = 2) +
        scale_color_manual(
            values = c("FALSE" = "black", "TRUE" = "red")
        ) +
        scale_x_continuous(limits = c(0, NA), breaks = c(0, break_vals)) +
        scale_y_continuous(limits = c(0, NA), breaks = c(0, break_vals)) +
        facet_wrap(vars(YEAR), nrow = 1) +
        labs(
            title = "INCIDENCE_ADJ_TESTING vs INCIDENCE_ADJ_CARESEEKING",
            subtitle = warning_text,
            caption = plot_caption
            ) +
        theme_minimal(base_size = 7) +
        theme(
            aspect.ratio = 1,
            legend.position = "none",
            strip.text = element_text(face = "bold", size = 10),
            panel.grid.minor = element_blank(),
            plot.caption = element_text(size = 7, hjust = 0)
        )

    plot_dir <- file.path(FIGURES_PATH, glue::glue("Incidence_year_adj_testing_vs_adj_careseeking_{DISAGGREGATION_SELECTION_SUFFIX}.png"))
    ggsave(
    filename = plot_dir, 
    plot = plot, 
    width = calc_width, 
    height = 21 / 2, 
    units = "cm", 
    dpi = 300
    )

    IRdisplay::display_png(file = plot_dir)

}

## Incidence du paludisme par année par district sanitaire

#### Puor annee et niveau d'ajustement

In [ ]:
# Step 1: Prepare long-form data
incidence_long <- yearly_incidence  %>% # incidence_data
  select(ADM2_ID, YEAR, POPULATION,
         INCIDENCE_CRUDE,
         INCIDENCE_ADJ_TESTING,
         INCIDENCE_ADJ_REPORTING,
         INCIDENCE_ADJ_CARESEEKING) %>%
  pivot_longer(
    cols = starts_with("INCIDENCE"),
    names_to = "INCIDENCE_TYPE",
    values_to = "incidence"
  ) %>%
  mutate(
    incidence_type_label = case_when(
      INCIDENCE_TYPE == "INCIDENCE_CRUDE"             ~ "Brute",
      INCIDENCE_TYPE == "INCIDENCE_ADJ_TESTING"       ~ "Ajustée 1\n(Test)",
      INCIDENCE_TYPE == "INCIDENCE_ADJ_REPORTING"     ~ "Ajustée 2\n(Test + Complétude)",
      INCIDENCE_TYPE == "INCIDENCE_ADJ_CARESEEKING"   ~ "Ajustée 3\n(Test + Complétude + Soins)",
      TRUE ~ INCIDENCE_TYPE
    )
  )

# Reorder incidence_type_label for plotting
incidence_long$incidence_type_label <- factor(
incidence_long$incidence_type_label,
levels = c("Brute", "Ajustée 1\n(Test)", "Ajustée 2\n(Test + Complétude)", "Ajustée 3\n(Test + Complétude + Soins)")
)
# # Remove INCIDENCE_ADJ_CARESEEKING if this is all empty ...
# filter(!is.na(incidence))


# Step 2: Join with shapefile
map_data_long <- shapes_data_stsimplified %>%
  left_join(incidence_long, by = "ADM2_ID")


# Step 3: categorize incidence based on break values from metadata
map_data_long <- map_data_long %>%
  mutate(
    INCIDENCE_CATEGORY = cut(
      incidence,
      breaks = full_breaks,
      labels = labels,
      right = TRUE, # so that 1.00 is assigned to "0.95 - 1.00"
      include.lowest = TRUE
    )
  )

In [ ]:
# Dynamically define subtitle text (handle `is.null(DISAGGREGATION_SELECTION)` so it disaplys TOTAL instead)
if (is.null(DISAGGREGATION_SELECTION)) {    
    subtitle_text <- "Brute et ajustée selon les étapes OMS.\nAucune désagrégation spécifique sélectionnée."
} else {
    subtitle_text <- glue::glue("Brute et ajustée selon les étapes OMS.\nDésagrégation utilisée: {DISAGGREGATION_SELECTION}.")
}

# Plot maps faceted by incidence type and year
plot <- ggplot(map_data_long) +
  geom_sf(aes(fill = INCIDENCE_CATEGORY), color = "white", size = 0.2) +
  facet_grid(
    rows = vars(incidence_type_label),
    cols = vars(YEAR)
    ) +
  scale_fill_manual(values = palette_to_use, name = "Incidence (pour 1000)") +
  labs(
    title = "Incidence annuelle du paludisme par district sanitaire",
    subtitle = subtitle_text,
    caption = plot_caption
  ) +
  theme_minimal(base_size = 10) +
  theme(
    strip.text = element_text(face = "bold", size = 9),
    plot.title = element_text(face = "bold", size = 10),
    plot.subtitle = element_text(size = 9),
    plot.caption = element_text(size = 7, hjust = 0),
    legend.position = "right",
    legend.justification = "top",
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    axis.text = element_blank(),
    axis.ticks = element_blank(),
  )

plot_dir <- file.path(FIGURES_PATH, glue::glue("Incidence_faceted_year_adjustment_{DISAGGREGATION_SELECTION_SUFFIX}.png"))
  ggsave(
    filename = plot_dir, 
    plot = plot, 
    width = calc_width + 10, 
    height = 31,
    units = "cm", 
    dpi = 300
    )

IRdisplay::display_png(file = plot_dir)


#### Moyenne annuelle (toutes années confondues)

In [ ]:
# Summarize incidence_long by computing mean incidence per INCIDENCE_TYPE and ADM2_ID across all years
incidence_long_mean <- incidence_long  %>% 
select(-POPULATION) %>%
# Added 20260128
group_by(ADM2_ID, INCIDENCE_TYPE,	incidence_type_label) |>
summarise(
  across(starts_with("INCIDENCE"), ~mean(., na.rm = TRUE)), # 🔍 pox PROBLEM here: if missing data for RR -> sum of N2 by YEAR is smaller than the sum of N1 !
  .groups = "drop"
        ) 

# Step 2: Join with shapefile
map_data_long_mean <- shapes_data_stsimplified %>%
  left_join(incidence_long_mean, by = "ADM2_ID")


# Step 3: categorize incidence based on break values from metadata
map_data_long_mean <- map_data_long_mean %>%
  mutate(
    INCIDENCE_CATEGORY = cut(
      incidence,
      breaks = full_breaks,
      labels = labels,
      right = TRUE, # so that 1.00 is assigned to "0.95 - 1.00"
      include.lowest = TRUE
    )
  )

# head(map_data_long_mean)

In [ ]:
subtitle_text_mean <- if (is.null(DISAGGREGATION_SELECTION)) {
    "Moyenne annuelle (toutes années confondues).\nAucune désagrégation spécifique sélectionnée."
} else {
    glue::glue("Moyenne annuelle (toutes années confondues).\nDésagrégation utilisée: {DISAGGREGATION_SELECTION}.")
}

# Plot maps faceted by incidence type
plot <- ggplot(map_data_long_mean) +
  geom_sf(aes(fill = INCIDENCE_CATEGORY), color = "white", size = 0.2) +
  facet_wrap(
    ~incidence_type_label,
    nrow = 1
    ) +
  scale_fill_manual(values = palette_to_use, name = "Incidence (pour 1000)") +
  labs(
    title = "Incidence moyenne du paludisme par district sanitaire",
    subtitle = subtitle_text_mean,
    caption = plot_caption
  ) +
  theme_minimal(base_size = 10) +
  theme(
    strip.text = element_text(face = "bold", size = 9),
    plot.title = element_text(face = "bold", size = 12),
    plot.subtitle = element_text(size = 10),
    plot.caption = element_text(size = 7, hjust = 0),
    legend.position = "right",
    legend.justification = "top",
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    axis.text = element_blank(),
    axis.ticks = element_blank(),
  )


# Export plots as png
YEAR_RANGE <- paste0(min(yearly_incidence$YEAR), "-", max(yearly_incidence$YEAR))

plot_dir <- file.path(FIGURES_PATH, glue::glue("Incidence_faceted_adjustment_{DISAGGREGATION_SELECTION_SUFFIX}_mean-{YEAR_RANGE}.png"))
  ggsave(
    filename = plot_dir, 
    plot = plot, 
    width = 45, 
    height = 15,
    units = "cm", 
    dpi = 300
    )

IRdisplay::display_png(file = plot_dir)